# Run the GTA tracking pipeline on Colab

Two-stage pipeline: **DeepEIoU tracking → GtaLink refinement**. Pipeline outputs (tracks, profiles,
eval) are written to Google Drive under `output/gta-track/<video>/`.

The **input clip** and the **rendered video** can come from / go to either **Google Drive** or
**Azure Blob Storage** (set `VIDEO_BACKEND` in step 1). 
Everything else — model checkpoints, ground truth, and pipeline artifacts — always uses Google Drive, which is always mounted.

Use a **GPU runtime**: *Runtime → Change runtime type → Hardware accelerator → GPU*.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Mount Drive & choose the video backend

`VIDEO_BACKEND` 
- `"drive"` — input from `…/input_videos/`; rendered video stays on Drive.
- `"azure"` — input downloaded from an Azure Blob container; rendered video uploaded back to Azure.

Then run the resolve cell (it mounts Drive and, for Azure, downloads the input clip).

In [ ]:
# ===== Config =====
VIDEO_BACKEND = "drive"   # "drive" or "azure"

# --- Input clip on Google Drive (used when VIDEO_BACKEND == "drive") ---
INPUT_VIDEO_NAME = "video2780-2960.mp4"     # file inside <DRIVE_BASE>/input_videos

# --- Azure Blob options (used when VIDEO_BACKEND == "azure") ---
import os
from getpass import getpass

# Auth: paste the connection string at the prompt, or pre-set the env var to skip it.
AZURE_CONNECTION_STRING = os.environ.get("AZURE_STORAGE_CONNECTION_STRING") or (
    getpass("Azure Storage connection string: ") if VIDEO_BACKEND == "azure" else ""
)

AZURE_INPUT_CONTAINER  = "test-videos"          # container holding the source clip
AZURE_INPUT_BLOB       = "test_video_second_half.mp4"    # blob path within that container

AZURE_OUTPUT_CONTAINER = "test-videos"       # container for the rendered output
AZURE_OUTPUT_PREFIX    = "gta-track"             # "folder" prefix within that container

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/football-analysis-project"
INPUT_DIR  = f"{DRIVE_BASE}/input_videos"
OUTPUT_DIR = f"{DRIVE_BASE}/output/gta-track"
CKPT_DIR   = f"{DRIVE_BASE}/checkpoints"
REPO       = "/content/football-analysis-2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if VIDEO_BACKEND == "drive":
    VIDEO = f"{INPUT_DIR}/{INPUT_VIDEO_NAME}"
    assert os.path.exists(VIDEO), f"input not found on Drive: {VIDEO}"

elif VIDEO_BACKEND == "azure":
    # Fetch the input clip from Azure into a local file (everything else stays on Drive).
    !pip install -q azure-storage-blob
    from azure.storage.blob import BlobServiceClient
    assert AZURE_CONNECTION_STRING, "set AZURE_CONNECTION_STRING in the config cell"
    _svc = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)

    def azure_download(container, blob, dest):
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest, "wb") as f:
            _svc.get_blob_client(container, blob).download_blob().readinto(f)
        return dest

    def azure_upload(container, blob, src):
        with open(src, "rb") as f:
            _svc.get_blob_client(container, blob).upload_blob(f, overwrite=True)
        return f"{container}/{blob}"

    os.makedirs("/content/input_videos", exist_ok=True)
    VIDEO = f"/content/input_videos/{Path(AZURE_INPUT_BLOB).name}"
    print(f"downloading {AZURE_INPUT_CONTAINER}/{AZURE_INPUT_BLOB} -> {VIDEO}")
    azure_download(AZURE_INPUT_CONTAINER, AZURE_INPUT_BLOB, VIDEO)

else:
    raise ValueError(f"unknown VIDEO_BACKEND: {VIDEO_BACKEND!r}")

STEM = Path(VIDEO).stem

# Export so the `!` shell cells below can read these:
os.environ.update(INPUT_DIR=INPUT_DIR, OUTPUT_DIR=OUTPUT_DIR,
                  CKPT_DIR=CKPT_DIR, REPO=REPO, VIDEO=VIDEO, STEM=STEM)

print("video backend:", VIDEO_BACKEND)
print("VIDEO        :", VIDEO)
print("STEM         :", STEM)
!ls -la "$INPUT_DIR"

## 2. Clone the code

In [ ]:
!git clone -b class-team-inference https://github.com/luna4tech/football-analysis-2.git "$REPO"
%cd $REPO

## 3. Install dependencies

One shared environment serves both stages. `torch`/`torchvision` are preinstalled on Colab. Install
`requirements.txt`, then `cython_bbox` (tracker IoU) + `tqdm` + `ultralytics`, then the **vendored** `reid`
(torchreid) package Stage 1 imports — install it from the repo, **not** `pip install torchreid`
(the PyPI layout is incompatible with the vendored code).

In [ ]:
%cd $REPO
!pip install -q -r gta-link/requirements.txt
!pip install -q cython_bbox tqdm ultralytics
!pip install -q -e Deep-EIoU/Deep-EIoU/reid

## 4. Download & link the model checkpoints

Fetches `sports_model.pth.tar-60` (ReID) into your Drive `checkpoints/` folder once,
then links it and your custom `yolov11l.pt` detector checkpoint where Stage 1 expects them.

In [ ]:
import glob, os

os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_FOLDER_URL = "https://drive.google.com/drive/folders/1wItcb0yeGaxOS08_G9yRWBTnpVf0vZ2w"

have = {os.path.basename(p) for p in glob.glob(f"{CKPT_DIR}/**/*", recursive=True)}
if "sports_model.pth.tar-60" not in have:
    !pip install -q gdown
    !gdown --folder "{CKPT_FOLDER_URL}" -O "$CKPT_DIR"

dst_dir = f"{REPO}/Deep-EIoU/Deep-EIoU/checkpoints"
os.makedirs(dst_dir, exist_ok=True)
for name in ("yolov11l.pt", "sports_model.pth.tar-60"):
    matches = glob.glob(f"{CKPT_DIR}/**/{name}", recursive=True)
    assert matches, f"{name} not found under {CKPT_DIR}"
    dst = f"{dst_dir}/{name}"
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(matches[0], dst)
    print("linked", matches[0], "->", dst)

!ls -la "{dst_dir}"

## 5. Run the pipeline

Runs Stage 1 (tracking) then Stage 2 (refine); outputs go to `OUTPUT_DIR/<STEM>/`.

In [ ]:
%cd $REPO
!python -m pipeline run \
    --video "$VIDEO" \
    --artifacts-dir "$OUTPUT_DIR" \
    --device gpu \
    --fp16 --fuse

Re-running is cached; to apply changed parameters add `--force-all` (or `--force-stage2`).

## 6. (Optional) Render an annotated video

The pipeline writes MOT `.txt`, not a video. Rebuild an overlay from the refined result (frames are
0-based, so no `--one_indexed`).

In [ ]:
%cd $REPO/Deep-EIoU/Deep-EIoU
!SAVE_PATH="$OUTPUT_DIR/$STEM/${STEM}_refined.mp4" && \
python tools/render_from_txt.py \
    --path "$VIDEO" \
    --txt  "$OUTPUT_DIR/$STEM/03_team/refined.txt" \
    --save_path "$SAVE_PATH" 
%cd $REPO

### 6b. Upload the rendered video to Azure

When `VIDEO_BACKEND == "azure"`, push the rendered clip (already written to Drive by step 6) to
`<AZURE_OUTPUT_CONTAINER>/<AZURE_OUTPUT_PREFIX>/<STEM>/<STEM>_refined.mp4`. For `"drive"` it's a
no-op — the file already lives on Drive.

In [ ]:
RENDERED = f"{OUTPUT_DIR}/{STEM}/{STEM}_refined.mp4"
assert os.path.exists(RENDERED), f"rendered video not found (run step 6 first): {RENDERED}"

if VIDEO_BACKEND == "azure":
    blob = f"{AZURE_OUTPUT_PREFIX}/{STEM}/{STEM}_refined.mp4"
    print(f"uploading {RENDERED} -> {AZURE_OUTPUT_CONTAINER}/{blob}")
    azure_upload(AZURE_OUTPUT_CONTAINER, blob, RENDERED)
    print("done")
else:
    print("Drive backend: rendered video already on Drive at", RENDERED)

## 7. Evaluate (HOTA / MOTA / IDF1)

The eval is **standalone and CPU-only** (it reuses this GPU session but needs no GPU). It scores the
pipeline's `refined.txt` against ground truth from Drive via **TrackEval**, reporting
**HOTA / DetA / AssA / MOTA / IDF1**.

The GT must be a **MOTChallenge `gt.txt`** (1-based frames); the pipeline's 0-based output is
converted automatically, and GT with `class=-1` is handled (no flag needed).

In [ ]:
!git clone -q https://github.com/JonathonLuiten/TrackEval.git /content/TrackEval
!pip install -q scipy

In [ ]:
GT_DIR = f"{DRIVE_BASE}/ground_truth"
GT = f"{GT_DIR}/gt_mot_{STEM}.txt"   # <-- EDIT to your GT file for THIS clip (MOTChallenge gt.txt)
os.environ["GT"] = GT
if not os.path.exists(GT):
    print("GT not found:", GT, "\nAvailable in", GT_DIR, ":")
    !ls -la "$GT_DIR" 2>/dev/null || echo "  (folder missing - create it and upload your gt.txt)"
    raise FileNotFoundError(GT)
print("Using GT:", GT)

In [ ]:
!grep -rl 'np\.float\|np\.int\|np\.bool' /content/TrackEval/trackeval | xargs -r sed -i 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g'

In [ ]:
%cd $REPO
# (--class-name defaults to 'pedestrian'; only override if your GT class label differs)
!python -m eval.evaluate \
    --pred "$OUTPUT_DIR/$STEM/03_team/refined.txt" \
    --gt   "$GT" \
    --seq-name "$STEM" \
    --trackeval-path /content/TrackEval \
    --out  "$OUTPUT_DIR/$STEM/eval/metrics.json"
import json
print(json.dumps(json.load(open(f"{OUTPUT_DIR}/{STEM}/eval/metrics.json")), indent=2))
print(json.dumps(json.load(open(f"{OUTPUT_DIR}/{STEM}/eval/attributes_metrics.json")), indent=2))

## Output layout

```
output/gta-track/<STEM>/
  01_track/tracks.txt       # raw tracking (MOT, 0-based frames)
  01_track/tracklets.pkl    # per-id tracklets with reused ReID features
  02_refine/refined.txt     # final refined result
  profiles/summary.md       # per-stage time / GPU / CPU / counts
  eval/metrics.json         # HOTA / DetA / AssA / MOTA / IDF1 (after step 7)
  <STEM>_refined.mp4        # only if you ran step 6
```

All of the above is written to Drive under `<DRIVE_BASE>/output/gta-track/`. When
`VIDEO_BACKEND == "azure"`, step 6b additionally uploads the rendered `<STEM>_refined.mp4` to
`<AZURE_OUTPUT_CONTAINER>/<AZURE_OUTPUT_PREFIX>/<STEM>/`.